# BMW Basic 2026 — Materi I
## CP2: deteksi R-Peak, hitung BPM, validasi ke nilai acuan

Target checkpoint (25 menit): BPM hasil hitunganmu berada dalam **±3 BPM** dari nilai di `data/reference_bpm.csv`, dan jumlah puncak masuk akal untuk rekaman 10 detik.

Di notebook ini kamu punya dua pilihan cara kerja:

1. **Isi `src/signal_utils.py`** seperti jalur utama, lalu notebook ini memakai fungsi itu. Ini yang disarankan kalau kamu kerja lokal.
2. **Tulis fungsimu langsung di notebook** (sel "ruang kerjamu" di bawah). Praktis di Colab, karena mengedit berkas di dalam repo hasil clone itu merepotkan.

Notebook otomatis memilih: kalau fungsi versi notebook masih kosong, ia jatuh ke versi `src/`.

In [ ]:
# Hanya untuk Google Colab (abaikan bila menjalankan lokal)
# !pip install -q plotly==5.22.0
# !git clone https://github.com/ZinniXX004/BMW2026-Basic-Python.git
# %cd BMW2026-Basic-Python
# !python data/make_dataset.py

In [ ]:
%matplotlib inline

import os
import pathlib
import sys

akar = pathlib.Path.cwd()
while not (akar / "data").is_dir() and akar != akar.parent:
    akar = akar.parent
os.chdir(akar)
sys.path.insert(0, str(akar))
print("Folder kerja:", akar)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

from src.signal_utils import (
    PERSENTIL_DEFAULT,
    REFRACTORY_DEFAULT,
    find_r_peaks,
    hitung_bpm,
    hitung_sdnn_ms,
    klasifikasi_hr,
    normalisasi_zscore,
)

FS = 250
print(f"Default materi: persentil={PERSENTIL_DEFAULT}, refractory={REFRACTORY_DEFAULT} s")

### Pilih sinyal yang mau dikerjakan

Mulailah dari EKG. PPG dipakai nanti, di bagian eksperimen parameter.

In [ ]:
PAKAI_PPG = False   # ubah jadi True kalau mau menguji sinyal PPG

berkas = "ppg_sample.csv" if PAKAI_PPG else "ecg_sample.csv"
kolom = "ppg_au" if PAKAI_PPG else "ecg_mv"
sinyal = pd.read_csv(f"data/{berkas}")[kolom].to_numpy()

acuan = pd.read_csv("data/reference_bpm.csv")
bpm_acuan = float(acuan.loc[acuan["file"] == berkas, "bpm_acuan"].iloc[0])

print(f"Berkas    : data/{berkas} ({sinyal.size} sampel = {sinyal.size / FS:.1f} s)")
print(f"BPM acuan : {bpm_acuan:.2f}")

---
## Ruang kerjamu (TODO CP2)

Tiga hal yang perlu kamu bangun, sama seperti TODO di `src/signal_utils.py`:

- **CP2-a** — ambang adaptif: normalisasi z-score, lalu ambil `np.percentile(x, persentil)` sebagai ambang. Kenapa persentil dan bukan angka tetap? Karena amplitudo tiap rekaman berbeda; ambang tetap yang cocok di satu pasien bisa gagal total di pasien lain.
- **CP2-b** — *refractory period*: setelah satu puncak diterima, tolak kandidat lain dalam `refractory_s` detik berikutnya. Ini yang mencegah gelombang T ikut terhitung sebagai denyut.
- **CP2-c** — BPM dari interval RR: `rr = np.diff(puncak) / fs`, lalu `60 / rr.mean()`. Kembalikan `np.nan` kalau puncaknya kurang dari dua — jangan pernah mengembalikan angka palsu.

Hapus `raise NotImplementedError` setelah kamu mengisi kodenya.

In [ ]:
def find_r_peaks_saya(sinyal, fs, persentil=95.0, refractory_s=0.25):
    """Kembalikan array indeks puncak. Isi TODO CP2-a dan CP2-b di sini."""
    raise NotImplementedError("CP2-a/b belum diisi")

    x = normalisasi_zscore(sinyal)

    # TODO (CP2-a): hitung ambang dari persentil
    # ambang = ...

    # TODO (CP2-b): cari maksimum lokal di atas ambang, hormati refractory period
    # jarak_min = int(refractory_s * fs)
    # puncak = []
    # for i in range(1, x.size - 1):
    #     ...
    # return np.asarray(puncak, dtype=int)


def hitung_bpm_saya(puncak, fs):
    """Kembalikan BPM rata-rata, atau np.nan kalau puncak < 2. Isi TODO CP2-c."""
    raise NotImplementedError("CP2-c belum diisi")

    # TODO (CP2-c)
    # if puncak.size < 2:
    #     return float("nan")
    # rr = ...
    # return ...

### Jalankan detektor

Sel ini memakai fungsi versi notebook kalau sudah kamu isi, dan versi `src/signal_utils.py` kalau belum. Perhatikan baris `Sumber fungsi` di keluaran — itu memberi tahu kode siapa yang sebenarnya sedang diuji.

In [ ]:
PERSENTIL = PERSENTIL_DEFAULT
REFRACTORY = REFRACTORY_DEFAULT

try:
    puncak = find_r_peaks_saya(sinyal, FS, persentil=PERSENTIL, refractory_s=REFRACTORY)
    sumber_puncak = "notebook (find_r_peaks_saya)"
except NotImplementedError:
    puncak = find_r_peaks(sinyal, FS, persentil=PERSENTIL, refractory_s=REFRACTORY)
    sumber_puncak = "src/signal_utils.py"

puncak = np.asarray(puncak, dtype=int)

try:
    bpm = hitung_bpm_saya(puncak, FS)
    sumber_bpm = "notebook (hitung_bpm_saya)"
except NotImplementedError:
    bpm = hitung_bpm(puncak, FS)
    sumber_bpm = "src/signal_utils.py"

sdnn = hitung_sdnn_ms(puncak, FS)

print(f"Sumber fungsi puncak : {sumber_puncak}")
print(f"Sumber fungsi BPM    : {sumber_bpm}")
print(f"Parameter            : persentil={PERSENTIL}, refractory={REFRACTORY} s")
print(f"Jumlah puncak        : {puncak.size}")
print(f"BPM                  : {bpm:.2f}")
print(f"SDNN                 : {sdnn:.2f} ms")
print(f"Label                : {klasifikasi_hr(bpm)}")

### Validasi ke nilai acuan

Di sinilah checkpoint dinilai. "Kodenya tidak error" bukan kriteria — angkanya harus benar.

In [ ]:
if np.isnan(bpm):
    print("BPM belum bisa dihitung. Periksa TODO CP2-a/b/c.")
else:
    selisih = abs(bpm - bpm_acuan)
    status = "LULUS" if selisih <= 3.0 else "BELUM LULUS"
    print(f"BPM hitung : {bpm:.2f}")
    print(f"BPM acuan  : {bpm_acuan:.2f}")
    print(f"Selisih    : {selisih:.2f} BPM -> {status}")

    if selisih > 3.0:
        print("\nPetunjuk urutan pemeriksaan:")
        print("  1. Puncak jauh lebih SEDIKIT dari denyut sebenarnya?")
        print("     Ambang persentil terlalu tinggi. Turunkan PERSENTIL.")
        print("  2. Puncak jauh lebih BANYAK? Refractory terlalu pendek,")
        print("     gelombang T ikut terhitung. Naikkan REFRACTORY.")
        print("  3. Jumlah puncak benar tapi BPM tetap salah? Periksa FS.")

### Bandingkan dengan pustaka

`scipy.signal.find_peaks` melakukan hal yang sama dalam satu baris. Implementasi manual tadi bukan pekerjaan sia-sia: tanpa pernah menulisnya sendiri, kamu tidak punya cara menilai apakah keluaran pustaka itu wajar.

Implementasi manual untuk paham; pustaka untuk pekerjaan nyata.

In [ ]:
x = normalisasi_zscore(sinyal)
pembanding, _ = find_peaks(
    x, height=np.percentile(x, PERSENTIL), distance=int(REFRACTORY * FS)
)

print(f"Punyamu : {puncak.size} puncak")
print(f"scipy   : {pembanding.size} puncak")

In [ ]:
if puncak.size:
    plt.figure(figsize=(11, 4))
    plt.plot(np.arange(sinyal.size) / FS, sinyal, linewidth=0.9, label=kolom)
    plt.plot(puncak / FS, sinyal[puncak], "o", markersize=5, label="Puncak")
    plt.title(f"Deteksi puncak, BPM = {bpm:.1f} (fs = {FS} Hz)")
    plt.xlabel("Waktu (s)")
    plt.ylabel("Amplitudo")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Belum ada puncak terdeteksi. Periksa kembali TODO CP2-a dan CP2-b.")

---
### Eksperimen: kenapa default-nya persentil 95, bukan 98

Ubah `PAKAI_PPG = True` di sel pilihan sinyal, jalankan ulang sampai sini, lalu jalankan sel di bawah. Persentil 98 lulus di berkas EKG tapi jatuh di PPG — **tanpa satu pun pesan error**.

Itulah alasan setiap detektor wajib divalidasi ke nilai acuan, bukan cuma dipastikan "tidak error".

In [ ]:
print(f"Berkas: data/{berkas}, acuan {bpm_acuan:.2f} BPM\n")
print(f"{'persentil':>10}{'puncak':>9}{'BPM':>9}{'selisih':>10}")

for p in (90.0, 95.0, 98.0):
    try:
        pk = np.asarray(find_r_peaks_saya(sinyal, FS, persentil=p, refractory_s=REFRACTORY), dtype=int)
    except NotImplementedError:
        pk = find_r_peaks(sinyal, FS, persentil=p, refractory_s=REFRACTORY)
    b = hitung_bpm(pk, FS)
    d = abs(b - bpm_acuan)
    print(f"{p:>10.0f}{pk.size:>9}{b:>9.2f}{d:>10.2f}")

### Daftar cek CP2

- [ ] `find_r_peaks` memakai ambang dari persentil, bukan angka tetap
- [ ] Refractory period dihormati (tidak ada dua puncak berjarak < 0,25 s)
- [ ] BPM mengembalikan `nan` kalau puncak < 2, bukan angka palsu
- [ ] Selisih ke acuan ≤ 3 BPM di `data/ecg_sample.csv`
- [ ] Sudah dicoba juga di `data/ppg_sample.csv`

### Setelah CP2

Lanjut ke CP3 (`src/app_dashboard.py`) lewat `streamlit run src/app_dashboard.py`. Dashboard Streamlit tidak bisa dijalankan dari dalam notebook — kalau environment lokalmu bermasalah, cukup kerjakan bagian perhitungannya dan lampirkan grafik matplotlib untuk KPP.

Mau memeriksa seluruh repo sekaligus?

```
!python tools/uji_cepat.py
!pip install -q pytest && python -m pytest -q
```